In [3]:
# 01_data_generation.ipynb
# Synthetic telematics-based auto insurance dataset
# Intentionally messy, multi-table, business + economics oriented

import os
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

# -----------------------------
# 1. Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker("en_US")
Faker.seed(SEED)

# -----------------------------
# 2. Paths
# -----------------------------
BASE_DIR = os.getcwd()
RAW_DIR = os.path.join(BASE_DIR, "data_raw")
os.makedirs(RAW_DIR, exist_ok=True)

# -----------------------------
# 3. Row targets
# -----------------------------
N_CUSTOMERS = 1100
TARGET_VEHICLES = 1200
TARGET_POLICIES = 1260
TARGET_TICKETS = 2400
TARGET_CLAIMS = 260

OBS_START = pd.Timestamp("2025-01-01")
OBS_END = pd.Timestamp("2025-12-31")

# -----------------------------
# 4. Category pools
# -----------------------------
states = [
    "California", "Texas", "Florida", "New York", "Illinois", "Georgia",
    "Arizona", "North Carolina", "Ohio", "Pennsylvania"
]

cities_by_state = {
    "California": ["Los Angeles", "San Diego", "Sacramento", "San Jose"],
    "Texas": ["Houston", "Dallas", "Austin", "San Antonio"],
    "Florida": ["Miami", "Orlando", "Tampa", "Jacksonville"],
    "New York": ["New York", "Buffalo", "Albany", "Rochester"],
    "Illinois": ["Chicago", "Springfield", "Aurora", "Naperville"],
    "Georgia": ["Atlanta", "Savannah", "Augusta", "Athens"],
    "Arizona": ["Phoenix", "Tucson", "Mesa", "Tempe"],
    "North Carolina": ["Charlotte", "Raleigh", "Durham", "Greensboro"],
    "Ohio": ["Columbus", "Cleveland", "Cincinnati", "Toledo"],
    "Pennsylvania": ["Philadelphia", "Pittsburgh", "Allentown", "Erie"]
}

income_bands = ["low", "lower_middle", "middle", "upper_middle", "high"]
income_probs = [0.15, 0.22, 0.30, 0.22, 0.11]

credit_tiers = ["poor", "fair", "good", "very_good", "excellent"]
credit_probs = [0.10, 0.18, 0.33, 0.25, 0.14]

marital_statuses = ["single", "married", "divorced", "widowed"]
marital_probs = [0.43, 0.42, 0.12, 0.03]

customer_segments = ["budget", "standard", "premium", "high_value"]
segment_probs = [0.25, 0.45, 0.22, 0.08]

vehicle_types = ["sedan", "suv", "pickup", "hatchback", "coupe", "minivan"]
vehicle_type_probs = [0.29, 0.27, 0.14, 0.12, 0.08, 0.10]

fuel_types = ["gasoline", "hybrid", "electric", "diesel"]
fuel_probs = [0.67, 0.16, 0.08, 0.09]

transmission_types = ["automatic", "manual"]
transmission_probs = [0.84, 0.16]

primary_uses = ["commuting", "family", "commercial_light", "leisure", "mixed"]
primary_use_probs = [0.37, 0.23, 0.08, 0.12, 0.20]

ownership_statuses = ["owned", "financed", "leased"]
ownership_probs = [0.44, 0.38, 0.18]

garage_types = ["garage", "street", "driveway", "covered_lot"]
garage_probs = [0.35, 0.28, 0.25, 0.12]

policy_types = ["liability_only", "standard", "premium", "full_coverage"]
policy_type_probs = [0.20, 0.38, 0.22, 0.20]

coverage_levels = ["basic", "enhanced", "comprehensive"]
coverage_probs = [0.33, 0.39, 0.28]

payment_frequencies = ["monthly", "quarterly", "semiannual", "annual"]
payment_probs = [0.57, 0.18, 0.15, 0.10]

support_channels = ["phone", "chat", "email", "self_service"]
channel_probs = [0.45, 0.28, 0.17, 0.10]

issue_types = [
    "app_login_issue", "mileage_sync_issue", "billing_issue",
    "discount_dispute", "policy_change_request", "claim_status_question",
    "payment_failure", "account_update_request"
]

claim_types = [
    "collision", "theft", "weather_damage", "glass_damage",
    "minor_property_damage", "bodily_injury", "roadside_assistance"
]

claim_severities = ["low", "medium", "high"]
claim_severity_probs = [0.58, 0.31, 0.11]

issue_severities = ["low", "medium", "high", "critical"]
issue_severity_probs = [0.40, 0.36, 0.18, 0.06]

vehicle_catalog = {
    "Toyota": [("Corolla", "sedan"), ("Camry", "sedan"), ("RAV4", "suv"), ("Tacoma", "pickup")],
    "Honda": [("Civic", "sedan"), ("Accord", "sedan"), ("CR-V", "suv"), ("Odyssey", "minivan")],
    "Ford": [("Focus", "hatchback"), ("Escape", "suv"), ("F-150", "pickup"), ("Mustang", "coupe")],
    "Chevrolet": [("Malibu", "sedan"), ("Equinox", "suv"), ("Silverado", "pickup"), ("Bolt", "hatchback")],
    "Nissan": [("Sentra", "sedan"), ("Altima", "sedan"), ("Rogue", "suv"), ("Leaf", "hatchback")],
    "Hyundai": [("Elantra", "sedan"), ("Sonata", "sedan"), ("Tucson", "suv"), ("Kona", "suv")],
    "Kia": [("Rio", "sedan"), ("Forte", "sedan"), ("Sportage", "suv"), ("Soul", "hatchback")],
    "Tesla": [("Model 3", "sedan"), ("Model Y", "suv")],
    "Jeep": [("Compass", "suv"), ("Grand Cherokee", "suv")],
    "Subaru": [("Impreza", "hatchback"), ("Outback", "suv"), ("Forester", "suv")]
}

# -----------------------------
# 5. Helper functions
# -----------------------------
def random_date(start: pd.Timestamp, end: pd.Timestamp) -> pd.Timestamp:
    delta_days = (end - start).days
    return start + pd.Timedelta(days=random.randint(0, delta_days))

def make_customer_id(i: int) -> str:
    return f"CUST{i:05d}"

def make_vehicle_id(i: int) -> str:
    return f"VEH{i:05d}"

def make_policy_id(i: int) -> str:
    return f"POL{i:05d}"

def make_ticket_id(i: int) -> str:
    return f"TIC{i:06d}"

def make_claim_id(i: int) -> str:
    return f"CLM{i:06d}"

def age_from_dob(dob: pd.Timestamp, reference: pd.Timestamp = OBS_END) -> int:
    return int((reference - dob).days // 365.25)

def weighted_choice(items, probs):
    return np.random.choice(items, p=probs)

def noisy_city(city: str) -> str:
    options = [
        city,
        city.lower(),
        city.upper(),
        f" {city}",
        f"{city} ",
        city.replace(" ", ""),
    ]
    return random.choice(options)

def noisy_label(value: str) -> str:
    options = [
        value,
        value.upper(),
        value.title(),
        value.replace("_", " "),
        f" {value}",
        f"{value} "
    ]
    return random.choice(options)

def calc_income_multiplier(band: str) -> float:
    mapping = {
        "low": 0.85,
        "lower_middle": 0.95,
        "middle": 1.00,
        "upper_middle": 1.12,
        "high": 1.28
    }
    return mapping.get(band, 1.0)

def calc_credit_multiplier(tier: str) -> float:
    mapping = {
        "poor": 1.20,
        "fair": 1.10,
        "good": 1.00,
        "very_good": 0.95,
        "excellent": 0.90
    }
    return mapping.get(tier, 1.0)

def vehicle_base_value(make: str, model: str, year: int) -> float:
    brand_bonus = {
        "Toyota": 24000, "Honda": 24500, "Ford": 25500, "Chevrolet": 25000,
        "Nissan": 23500, "Hyundai": 22500, "Kia": 22000, "Tesla": 42000,
        "Jeep": 32000, "Subaru": 28000
    }.get(make, 25000)
    age_penalty = max(0, (2025 - year) * 1400)
    value = brand_bonus - age_penalty + np.random.normal(0, 3500)
    return max(3500, round(value, 2))

def risk_from_profile(age, annual_mileage, driving_score, night_pct, speeding_events, hard_brakes, claim_hist=0):
    risk = 0.45
    if age < 25:
        risk += 0.18
    elif age > 65:
        risk += 0.05
    risk += min(annual_mileage / 30000, 1.3) * 0.18
    risk += max(0, (70 - driving_score)) * 0.004
    risk += night_pct * 0.15
    risk += min(speeding_events / 40, 1) * 0.18
    risk += min(hard_brakes / 35, 1) * 0.14
    risk += claim_hist * 0.10
    return round(min(max(risk, 0.20), 1.60), 3)

def premium_from_risk(vehicle_value, coverage_level, policy_type, risk_score, credit_tier):
    coverage_mult = {"basic": 1.00, "enhanced": 1.18, "comprehensive": 1.36}[coverage_level]
    policy_mult = {"liability_only": 0.82, "standard": 1.00, "premium": 1.18, "full_coverage": 1.30}[policy_type]
    credit_mult = calc_credit_multiplier(credit_tier)
    base = 45 + (vehicle_value * 0.0024)
    premium = base * coverage_mult * policy_mult * risk_score * credit_mult
    premium += np.random.normal(0, 12)
    return round(max(40, premium), 2)

def support_cost_from_ticket(severity, channel, resolution_hours):
    sev_cost = {"low": 8, "medium": 16, "high": 32, "critical": 55}[severity]
    channel_cost = {"phone": 12, "chat": 8, "email": 6, "self_service": 2}[channel]
    time_cost = min(resolution_hours * 0.8, 70)
    return round(sev_cost + channel_cost + time_cost, 2)

# -----------------------------
# 6. Generate customers
# -----------------------------
customers = []
for i in range(1, N_CUSTOMERS + 1):
    customer_id = make_customer_id(i)
    gender = random.choice(["M", "F"])
    first_name = fake.first_name_male() if gender == "M" else fake.first_name_female()
    last_name = fake.last_name()

    dob = pd.Timestamp(fake.date_of_birth(minimum_age=18, maximum_age=78))
    age = age_from_dob(dob)

    state = random.choice(states)
    city = random.choice(cities_by_state[state])

    join_date = random_date(pd.Timestamp("2022-01-01"), pd.Timestamp("2025-10-31"))
    tenure_months = max(1, ((OBS_END.year - join_date.year) * 12 + (OBS_END.month - join_date.month)))

    income_band = weighted_choice(income_bands, income_probs)
    credit_tier = weighted_choice(credit_tiers, credit_probs)
    marital_status = weighted_choice(marital_statuses, marital_probs)
    customer_segment = weighted_choice(customer_segments, segment_probs)

    row = {
        "customer_id": customer_id,
        "first_name": first_name,
        "last_name": last_name,
        "gender": gender,
        "date_of_birth": dob,
        "age": age,
        "email": fake.email(),
        "phone_number": fake.phone_number(),
        "state": state,
        "city": city,
        "zip_code_group": str(fake.zipcode())[:3],
        "income_band": income_band,
        "credit_tier": credit_tier,
        "marital_status": marital_status,
        "household_size": np.random.choice([1, 2, 3, 4, 5, 6], p=[0.24, 0.28, 0.18, 0.16, 0.10, 0.04]),
        "customer_join_date": join_date,
        "tenure_months": tenure_months,
        "customer_segment": customer_segment
    }
    customers.append(row)

customers_df = pd.DataFrame(customers)

# Inject mess into customers
# Missing emails
mask = np.random.rand(len(customers_df)) < 0.06
customers_df.loc[mask, "email"] = None

# Inconsistent city formatting
mask = np.random.rand(len(customers_df)) < 0.18
customers_df.loc[mask, "city"] = customers_df.loc[mask, "city"].apply(noisy_city)

# Missing income band
mask = np.random.rand(len(customers_df)) < 0.04
customers_df.loc[mask, "income_band"] = None

# Weird phone formatting
mask = np.random.rand(len(customers_df)) < 0.12
customers_df.loc[mask, "phone_number"] = customers_df.loc[mask, "phone_number"].apply(
    lambda x: x.replace("-", "").replace("(", "").replace(")", "") if isinstance(x, str) else x
)

# Duplicate-looking names
dup_idx = np.random.choice(customers_df.index, size=12, replace=False)
src_idx = np.random.choice(customers_df.index, size=12, replace=False)
customers_df.loc[dup_idx, ["first_name", "last_name"]] = customers_df.loc[src_idx, ["first_name", "last_name"]].values

# -----------------------------
# 7. Generate vehicles
# -----------------------------
vehicle_rows = []
vehicle_counter = 1

# Vehicle count per customer
# Majority 1, some 2, very few 3
vehicle_counts = np.random.choice([1, 2, 3], size=N_CUSTOMERS, p=[0.89, 0.10, 0.01])

for customer_id, n_veh in zip(customers_df["customer_id"], vehicle_counts):
    for _ in range(n_veh):
        make = random.choice(list(vehicle_catalog.keys()))
        model, implied_type = random.choice(vehicle_catalog[make])
        vehicle_year = random.randint(2010, 2025)
        vehicle_type = implied_type if np.random.rand() < 0.82 else weighted_choice(vehicle_types, vehicle_type_probs)
        annual_mileage = int(max(2500, np.random.normal(12500, 5000)))
        value_estimate = vehicle_base_value(make, model, vehicle_year)

        vehicle_rows.append({
            "vehicle_id": make_vehicle_id(vehicle_counter),
            "customer_id": customer_id,
            "vehicle_make": make,
            "vehicle_model": model,
            "vehicle_year": vehicle_year,
            "vehicle_type": vehicle_type,
            "fuel_type": weighted_choice(fuel_types, fuel_probs),
            "transmission_type": weighted_choice(transmission_types, transmission_probs),
            "vehicle_value_estimate": value_estimate,
            "annual_mileage": annual_mileage,
            "primary_use": weighted_choice(primary_uses, primary_use_probs),
            "ownership_status": weighted_choice(ownership_statuses, ownership_probs),
            "garage_type": weighted_choice(garage_types, garage_probs),
            "anti_theft_device_flag": np.random.choice([0, 1], p=[0.68, 0.32])
        })
        vehicle_counter += 1

vehicles_df = pd.DataFrame(vehicle_rows)

# Keep around target size
if len(vehicles_df) > TARGET_VEHICLES:
    vehicles_df = vehicles_df.sample(TARGET_VEHICLES, random_state=SEED).sort_values("vehicle_id").reset_index(drop=True)

# Messiness in vehicles
mask = np.random.rand(len(vehicles_df)) < 0.07
vehicles_df.loc[mask, "garage_type"] = None

mask = np.random.rand(len(vehicles_df)) < 0.10
vehicles_df.loc[mask, "vehicle_make"] = vehicles_df.loc[mask, "vehicle_make"].apply(noisy_label)

mask = np.random.rand(len(vehicles_df)) < 0.08
vehicles_df.loc[mask, "vehicle_type"] = vehicles_df.loc[mask, "vehicle_type"].apply(noisy_label)

# Outliers in vehicle value
outlier_idx = np.random.choice(vehicles_df.index, size=8, replace=False)
vehicles_df.loc[outlier_idx, "vehicle_value_estimate"] *= np.random.choice([0.35, 2.7], size=8)

# -----------------------------
# 8. Generate policies
# -----------------------------
# Most vehicles get one policy, some get renewals / replacements
policy_rows = []
policy_counter = 1

policy_vehicle_ids = vehicles_df["vehicle_id"].tolist()
base_vehicle_sample = policy_vehicle_ids.copy()

# First pass: at least one policy per vehicle until target is reached
for veh_id in base_vehicle_sample:
    if policy_counter > TARGET_POLICIES:
        break

    vehicle_row = vehicles_df.loc[vehicles_df["vehicle_id"] == veh_id].iloc[0]
    customer_row = customers_df.loc[customers_df["customer_id"] == vehicle_row["customer_id"]].iloc[0]

    start_date = random_date(pd.Timestamp("2024-01-01"), pd.Timestamp("2025-09-30"))
    end_date = start_date + pd.Timedelta(days=364)

    policy_type = weighted_choice(policy_types, policy_type_probs)
    coverage_level = weighted_choice(coverage_levels, coverage_probs)

    # Temporary driving profile proxy for pricing
    proxy_driving_score = int(np.clip(np.random.normal(76, 12), 35, 100))
    proxy_night_pct = round(np.clip(np.random.beta(2.0, 8.0), 0, 1), 3)
    proxy_speeding = max(0, int(np.random.poisson(6)))
    proxy_hard_brakes = max(0, int(np.random.poisson(8)))

    risk_score = risk_from_profile(
        age=customer_row["age"],
        annual_mileage=vehicle_row["annual_mileage"],
        driving_score=proxy_driving_score,
        night_pct=proxy_night_pct,
        speeding_events=proxy_speeding,
        hard_brakes=proxy_hard_brakes,
        claim_hist=np.random.choice([0, 1, 2], p=[0.74, 0.20, 0.06])
    )

    deductible = np.random.choice([250, 500, 1000, 1500], p=[0.18, 0.43, 0.31, 0.08])

    base_premium = premium_from_risk(
        vehicle_value=float(vehicle_row["vehicle_value_estimate"]),
        coverage_level=coverage_level,
        policy_type=policy_type,
        risk_score=risk_score,
        credit_tier=customer_row["credit_tier"] if pd.notna(customer_row["credit_tier"]) else "good"
    )

    telematics_discount_pct = round(np.clip((proxy_driving_score - 50) / 3.4, 0, 18), 2)
    loyalty_discount_pct = round(np.random.choice([0, 0, 0, 3, 5, 7], p=[0.22, 0.18, 0.12, 0.20, 0.18, 0.10]), 2)
    bundled_discount_pct = round(np.random.choice([0, 0, 4, 6, 8], p=[0.46, 0.18, 0.18, 0.12, 0.06]), 2)

    total_discount_pct = min(telematics_discount_pct + loyalty_discount_pct + bundled_discount_pct, 28)
    final_monthly_premium = round(base_premium * (1 - total_discount_pct / 100), 2)

    policy_status = np.random.choice(["active", "renewed", "cancelled", "lapsed"], p=[0.52, 0.18, 0.18, 0.12])
    renewal_flag = int(policy_status == "renewed")
    churn_flag = int(policy_status in ["cancelled", "lapsed"])

    cancellation_reason = None
    if churn_flag == 1:
        cancellation_reason = np.random.choice(
            ["price_increase", "moved_to_competitor", "service_issues", "billing_problems", "claim_dispute", None],
            p=[0.30, 0.26, 0.18, 0.12, 0.08, 0.06]
        )

    policy_rows.append({
        "policy_id": make_policy_id(policy_counter),
        "customer_id": vehicle_row["customer_id"],
        "vehicle_id": veh_id,
        "policy_start_date": start_date,
        "policy_end_date": end_date,
        "policy_type": policy_type,
        "coverage_level": coverage_level,
        "deductible": deductible,
        "risk_score": risk_score,
        "base_premium": base_premium,
        "telematics_discount_pct": telematics_discount_pct,
        "loyalty_discount_pct": loyalty_discount_pct,
        "bundled_discount_pct": bundled_discount_pct,
        "final_monthly_premium": final_monthly_premium,
        "payment_frequency": weighted_choice(payment_frequencies, payment_probs),
        "autopay_flag": np.random.choice([0, 1], p=[0.42, 0.58]),
        "policy_status": policy_status,
        "renewal_flag": renewal_flag,
        "churn_flag": churn_flag,
        "cancellation_reason": cancellation_reason,
        "payment_timeliness_score": round(np.clip(np.random.normal(82, 15), 20, 100), 1)
    })
    policy_counter += 1

# Add additional policies to hit target
while policy_counter <= TARGET_POLICIES:
    veh_id = random.choice(policy_vehicle_ids)
    vehicle_row = vehicles_df.loc[vehicles_df["vehicle_id"] == veh_id].iloc[0]
    customer_row = customers_df.loc[customers_df["customer_id"] == vehicle_row["customer_id"]].iloc[0]

    start_date = random_date(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-12-15"))
    end_date = start_date + pd.Timedelta(days=364)

    policy_type = weighted_choice(policy_types, policy_type_probs)
    coverage_level = weighted_choice(coverage_levels, coverage_probs)
    driving_score = int(np.clip(np.random.normal(74, 13), 30, 100))
    night_pct = round(np.clip(np.random.beta(2.1, 7.4), 0, 1), 3)
    speeding_events = max(0, int(np.random.poisson(7)))
    hard_brakes = max(0, int(np.random.poisson(9)))

    risk_score = risk_from_profile(
        customer_row["age"],
        vehicle_row["annual_mileage"],
        driving_score,
        night_pct,
        speeding_events,
        hard_brakes,
        claim_hist=np.random.choice([0, 1, 2], p=[0.73, 0.20, 0.07])
    )

    base_premium = premium_from_risk(
        float(vehicle_row["vehicle_value_estimate"]),
        coverage_level,
        policy_type,
        risk_score,
        customer_row["credit_tier"] if pd.notna(customer_row["credit_tier"]) else "good"
    )

    telematics_discount_pct = round(np.clip((driving_score - 52) / 3.8, 0, 18), 2)
    loyalty_discount_pct = round(np.random.choice([0, 3, 5, 7], p=[0.57, 0.20, 0.16, 0.07]), 2)
    bundled_discount_pct = round(np.random.choice([0, 4, 6, 8], p=[0.70, 0.18, 0.09, 0.03]), 2)
    total_discount_pct = min(telematics_discount_pct + loyalty_discount_pct + bundled_discount_pct, 28)
    final_monthly_premium = round(base_premium * (1 - total_discount_pct / 100), 2)

    policy_status = np.random.choice(["active", "renewed", "cancelled", "lapsed"], p=[0.44, 0.20, 0.22, 0.14])
    renewal_flag = int(policy_status == "renewed")
    churn_flag = int(policy_status in ["cancelled", "lapsed"])

    policy_rows.append({
        "policy_id": make_policy_id(policy_counter),
        "customer_id": vehicle_row["customer_id"],
        "vehicle_id": veh_id,
        "policy_start_date": start_date,
        "policy_end_date": end_date,
        "policy_type": policy_type,
        "coverage_level": coverage_level,
        "deductible": np.random.choice([250, 500, 1000, 1500], p=[0.20, 0.45, 0.27, 0.08]),
        "risk_score": risk_score,
        "base_premium": base_premium,
        "telematics_discount_pct": telematics_discount_pct,
        "loyalty_discount_pct": loyalty_discount_pct,
        "bundled_discount_pct": bundled_discount_pct,
        "final_monthly_premium": final_monthly_premium,
        "payment_frequency": weighted_choice(payment_frequencies, payment_probs),
        "autopay_flag": np.random.choice([0, 1], p=[0.41, 0.59]),
        "policy_status": policy_status,
        "renewal_flag": renewal_flag,
        "churn_flag": churn_flag,
        "cancellation_reason": np.random.choice(
            ["price_increase", "moved_to_competitor", "service_issues", "billing_problems", "claim_dispute", None],
            p=[0.28, 0.22, 0.17, 0.12, 0.09, 0.12]
        ) if churn_flag == 1 else None,
        "payment_timeliness_score": round(np.clip(np.random.normal(81, 16), 15, 100), 1)
    })
    policy_counter += 1

policies_df = pd.DataFrame(policy_rows)

# Messiness in policies
mask = np.random.rand(len(policies_df)) < 0.08
policies_df.loc[mask, "policy_status"] = policies_df.loc[mask, "policy_status"].apply(noisy_label)

mask = np.random.rand(len(policies_df)) < 0.06
policies_df.loc[mask, "telematics_discount_pct"] = None

mask = np.random.rand(len(policies_df)) < 0.05
policies_df.loc[mask, "bundled_discount_pct"] = None

# suspicious premiums
idx = np.random.choice(policies_df.index, size=10, replace=False)
policies_df.loc[idx, "final_monthly_premium"] *= np.random.choice([0.35, 2.4], size=10)

# -----------------------------
# 9. Generate driving behavior
# -----------------------------
driving_rows = []
driving_counter = 1

months_2025 = pd.date_range("2025-01-01", "2025-12-01", freq="MS")

policy_lookup = policies_df[["policy_id", "customer_id", "vehicle_id", "risk_score", "telematics_discount_pct"]].copy()
vehicle_lookup = vehicles_df.set_index("vehicle_id")
customer_lookup = customers_df.set_index("customer_id")

for _, pol in policy_lookup.iterrows():
    vehicle = vehicle_lookup.loc[pol["vehicle_id"]]
    customer = customer_lookup.loc[pol["customer_id"]]

    # Not every policy gets all 12 months of logging
    n_months = np.random.choice([6, 8, 10, 12], p=[0.10, 0.18, 0.24, 0.48])
    chosen_months = sorted(np.random.choice(months_2025, size=n_months, replace=False))

    for month in chosen_months:
        age = customer["age"]
        base_score = 82 - (pol["risk_score"] - 0.5) * 18 + np.random.normal(0, 8)
        driving_score = int(np.clip(base_score, 25, 100))
        trips_per_week = max(1, int(np.random.normal(7.5, 2.8)))
        avg_trip_distance = round(max(1.5, np.random.normal(11, 5)), 2)
        hard_brakes = max(0, int(np.random.poisson(7 if driving_score > 70 else 12)))
        rapid_accel = max(0, int(np.random.poisson(6 if driving_score > 70 else 10)))
        night_pct = round(np.clip(np.random.beta(2.3, 7.5), 0, 1), 3)
        weekend_pct = round(np.clip(np.random.beta(3.3, 4.4), 0, 1), 3)
        speeding_events = max(0, int(np.random.poisson(4 if driving_score > 75 else 9)))
        urban_pct = round(np.clip(np.random.beta(4.8, 3.0), 0, 1), 3)
        highway_pct = round(max(0, min(1, 1 - urban_pct + np.random.normal(0, 0.08))), 3)
        mileage_logged = int(max(50, np.random.normal(vehicle["annual_mileage"] / 12, 240)))

        driving_rows.append({
            "driving_record_id": f"DRV{driving_counter:06d}",
            "policy_id": pol["policy_id"],
            "customer_id": pol["customer_id"],
            "vehicle_id": pol["vehicle_id"],
            "observation_month": pd.Timestamp(month),
            "driving_score": driving_score,
            "avg_trip_distance": avg_trip_distance,
            "trips_per_week": trips_per_week,
            "hard_brakes_count": hard_brakes,
            "rapid_acceleration_count": rapid_accel,
            "night_driving_pct": night_pct,
            "weekend_driving_pct": weekend_pct,
            "distracted_driving_flag": np.random.choice([0, 1], p=[0.82, 0.18]),
            "speeding_events_count": speeding_events,
            "urban_driving_pct": urban_pct,
            "highway_driving_pct": highway_pct,
            "mileage_logged": mileage_logged
        })
        driving_counter += 1

driving_df = pd.DataFrame(driving_rows)

# Messiness in driving data
mask = np.random.rand(len(driving_df)) < 0.05
driving_df.loc[mask, "mileage_logged"] = None

mask = np.random.rand(len(driving_df)) < 0.03
driving_df.loc[mask, "observation_month"] = driving_df.loc[mask, "observation_month"].astype(str).str.replace("-01", "", regex=False)

outlier_idx = np.random.choice(driving_df.index, size=20, replace=False)
driving_df.loc[outlier_idx, "driving_score"] = np.random.choice([5, 12, 140, 180], size=20)

# -----------------------------
# 10. Generate support tickets
# -----------------------------
ticket_rows = []
ticket_counter = 1

policy_ids = policies_df["policy_id"].tolist()

# customer-level support propensity
customer_ticket_weight = {}
for _, row in customers_df.iterrows():
    base = 0.7
    if row["customer_segment"] == "budget":
        base += 0.15
    if row["customer_segment"] == "high_value":
        base += 0.25
    customer_ticket_weight[row["customer_id"]] = base

policy_map = policies_df.set_index("policy_id")

while ticket_counter <= TARGET_TICKETS:
    policy_id = random.choice(policy_ids)
    pol = policy_map.loc[policy_id]
    customer_id = pol["customer_id"]

    if np.random.rand() > min(customer_ticket_weight[customer_id] / 2.0, 0.92):
        continue

    open_date = random_date(OBS_START, OBS_END)
    severity = weighted_choice(issue_severities, issue_severity_probs)
    channel = weighted_choice(support_channels, channel_probs)

    base_res_time = {"low": 6, "medium": 18, "high": 44, "critical": 90}[severity]
    resolution_hours = round(max(0.5, np.random.normal(base_res_time, base_res_time * 0.55)), 2)

    resolution_status = np.random.choice(["resolved", "pending", "closed_no_action", "escalated"], p=[0.63, 0.15, 0.08, 0.14])
    close_date = open_date + pd.Timedelta(hours=float(resolution_hours)) if resolution_status in ["resolved", "closed_no_action"] else pd.NaT

    csat = None
    if resolution_status == "resolved":
        csat = np.random.choice([1, 2, 3, 4, 5], p=[0.10, 0.12, 0.18, 0.28, 0.32])
    elif resolution_status == "escalated":
        csat = np.random.choice([1, 2, 3, None], p=[0.35, 0.30, 0.20, 0.15])

    refund_flag = np.random.choice([0, 1], p=[0.84, 0.16])
    refund_amt = round(max(0, np.random.normal(18, 22)), 2) if refund_flag == 1 else 0.0

    ticket_rows.append({
        "ticket_id": make_ticket_id(ticket_counter),
        "customer_id": customer_id,
        "policy_id": policy_id,
        "ticket_open_date": open_date,
        "ticket_close_date": close_date,
        "issue_type": random.choice(issue_types),
        "issue_severity": severity,
        "support_channel": channel,
        "resolution_status": resolution_status,
        "resolution_time_hours": resolution_hours,
        "escalation_flag": int(resolution_status == "escalated"),
        "repeat_ticket_flag": np.random.choice([0, 1], p=[0.76, 0.24]),
        "csat_score": csat,
        "refund_or_credit_given_flag": refund_flag,
        "refund_or_credit_amount": refund_amt
    })
    ticket_counter += 1

tickets_df = pd.DataFrame(ticket_rows)

# Messiness in tickets
mask = np.random.rand(len(tickets_df)) < 0.08
tickets_df.loc[mask, "issue_type"] = tickets_df.loc[mask, "issue_type"].apply(noisy_label)

mask = np.random.rand(len(tickets_df)) < 0.05
tickets_df.loc[mask, "csat_score"] = None

mask = np.random.rand(len(tickets_df)) < 0.03
tickets_df.loc[mask, "refund_or_credit_amount"] = None

# duplicate-ish tickets
dup_sample = tickets_df.sample(18, random_state=SEED).copy()
dup_sample["ticket_id"] = [make_ticket_id(TARGET_TICKETS + i + 1) for i in range(len(dup_sample))]
dup_sample["ticket_open_date"] = dup_sample["ticket_open_date"] + pd.to_timedelta(np.random.randint(0, 3, len(dup_sample)), unit="h")
tickets_df = pd.concat([tickets_df, dup_sample], ignore_index=True)

# -----------------------------
# 11. Generate claims
# -----------------------------
claim_rows = []
claim_counter = 1

# Choose claims from policies with probability based on risk
policy_claim_probs = policies_df[["policy_id", "customer_id", "vehicle_id", "risk_score", "coverage_level"]].copy()
prob = policy_claim_probs["risk_score"].clip(lower=0.25, upper=1.60)
prob = prob / prob.sum()

selected_policies = np.random.choice(
    policy_claim_probs["policy_id"],
    size=TARGET_CLAIMS,
    replace=True,
    p=prob
)

for policy_id in selected_policies:
    pol = policies_df.loc[policies_df["policy_id"] == policy_id].iloc[0]
    vehicle = vehicles_df.loc[vehicles_df["vehicle_id"] == pol["vehicle_id"]].iloc[0]

    claim_type = random.choice(claim_types)
    severity = weighted_choice(claim_severities, claim_severity_probs)

    if severity == "low":
        estimated_cost = round(max(120, np.random.normal(900, 450)), 2)
    elif severity == "medium":
        estimated_cost = round(max(700, np.random.normal(4200, 1600)), 2)
    else:
        estimated_cost = round(max(3500, np.random.normal(14500, 5200)), 2)

    claim_status = np.random.choice(["approved", "pending", "rejected", "under_review"], p=[0.56, 0.18, 0.11, 0.15])

    approved_amount = None
    if claim_status == "approved":
        approved_amount = round(max(80, estimated_cost * np.random.uniform(0.65, 1.00)), 2)
    elif claim_status == "rejected":
        approved_amount = 0.0

    claim_rows.append({
        "claim_id": make_claim_id(claim_counter),
        "customer_id": pol["customer_id"],
        "policy_id": policy_id,
        "vehicle_id": pol["vehicle_id"],
        "claim_date": random_date(OBS_START, OBS_END),
        "claim_type": claim_type,
        "claim_severity": severity,
        "estimated_claim_cost": estimated_cost,
        "approved_claim_amount": approved_amount,
        "fraud_risk_flag": np.random.choice([0, 1], p=[0.90, 0.10]),
        "claim_status": claim_status,
        "fault_assignment": np.random.choice(["insured", "third_party", "shared", "unknown"], p=[0.46, 0.20, 0.16, 0.18]),
        "weather_related_flag": np.random.choice([0, 1], p=[0.78, 0.22])
    })
    claim_counter += 1

claims_df = pd.DataFrame(claim_rows)

# Messiness in claims
mask = np.random.rand(len(claims_df)) < 0.06
claims_df.loc[mask, "claim_status"] = claims_df.loc[mask, "claim_status"].apply(noisy_label)

outlier_idx = np.random.choice(claims_df.index, size=6, replace=False)
claims_df.loc[outlier_idx, "estimated_claim_cost"] *= np.random.choice([0.20, 3.80], size=6)

# contradictory rows
contr_idx = np.random.choice(claims_df.index, size=5, replace=False)
claims_df.loc[contr_idx, "claim_status"] = np.random.choice(["approved", "rejected"], size=5)
claims_df.loc[contr_idx, "approved_claim_amount"] = np.random.choice([None, 0, 15000], size=5)

# -----------------------------
# 12. Financial summary (derived)
# -----------------------------
# Revenue proxy: monthly premium times estimated active months in 2025
policies_tmp = policies_df.copy()
policies_tmp["policy_start_date"] = pd.to_datetime(policies_tmp["policy_start_date"], errors="coerce")
policies_tmp["policy_end_date"] = pd.to_datetime(policies_tmp["policy_end_date"], errors="coerce")

def active_months_in_2025(start_date, end_date):
    if pd.isna(start_date) or pd.isna(end_date):
        return 0
    start = max(start_date, pd.Timestamp("2025-01-01"))
    end = min(end_date, pd.Timestamp("2025-12-31"))
    if end < start:
        return 0
    return max(1, ((end.year - start.year) * 12 + (end.month - start.month) + 1))

policies_tmp["active_months_2025"] = policies_tmp.apply(
    lambda r: active_months_in_2025(r["policy_start_date"], r["policy_end_date"]), axis=1
)

# Support cost
tickets_tmp = tickets_df.copy()
tickets_tmp["ticket_support_cost"] = tickets_tmp.apply(
    lambda r: support_cost_from_ticket(
        severity=str(r["issue_severity"]).strip().lower() if pd.notna(r["issue_severity"]) else "medium",
        channel=str(r["support_channel"]).strip().lower() if pd.notna(r["support_channel"]) else "email",
        resolution_hours=float(r["resolution_time_hours"]) if pd.notna(r["resolution_time_hours"]) else 8.0
    ),
    axis=1
)

support_cost_by_policy = tickets_tmp.groupby("policy_id", as_index=False)["ticket_support_cost"].sum()
claims_paid_by_policy = claims_df.groupby("policy_id", as_index=False)["approved_claim_amount"].sum(min_count=1)

financial_df = policies_tmp.merge(support_cost_by_policy, on="policy_id", how="left")
financial_df = financial_df.merge(claims_paid_by_policy, on="policy_id", how="left")

financial_df["ticket_support_cost"] = financial_df["ticket_support_cost"].fillna(0)
financial_df["approved_claim_amount"] = financial_df["approved_claim_amount"].fillna(0)

financial_df["total_premium_paid"] = financial_df["final_monthly_premium"] * financial_df["active_months_2025"]
financial_df["total_discount_amount"] = (
    (financial_df["base_premium"] - financial_df["final_monthly_premium"]).clip(lower=0)
    * financial_df["active_months_2025"]
)

financial_df["estimated_customer_ltv"] = (
    financial_df["total_premium_paid"] * np.where(financial_df["renewal_flag"] == 1, 1.55, 1.0)
)

financial_df["estimated_net_value"] = (
    financial_df["total_premium_paid"]
    - financial_df["ticket_support_cost"]
    - financial_df["approved_claim_amount"]
)

financial_df["profitability_segment"] = pd.cut(
    financial_df["estimated_net_value"],
    bins=[-1e9, 0, 500, 2000, 1e9],
    labels=["negative", "low", "medium", "high"]
).astype(str)

financial_df = financial_df[
    [
        "customer_id", "policy_id", "vehicle_id", "total_premium_paid",
        "total_discount_amount", "ticket_support_cost", "approved_claim_amount",
        "estimated_customer_ltv", "estimated_net_value", "profitability_segment"
    ]
].rename(columns={
    "ticket_support_cost": "total_support_cost",
    "approved_claim_amount": "total_claim_cost"
})

# -----------------------------
# 13. Save CSVs
# -----------------------------
customers_df.to_csv(os.path.join(RAW_DIR, "customers.csv"), index=False)
vehicles_df.to_csv(os.path.join(RAW_DIR, "vehicles.csv"), index=False)
policies_df.to_csv(os.path.join(RAW_DIR, "policies.csv"), index=False)
driving_df.to_csv(os.path.join(RAW_DIR, "driving_behavior.csv"), index=False)
tickets_df.to_csv(os.path.join(RAW_DIR, "support_tickets.csv"), index=False)
claims_df.to_csv(os.path.join(RAW_DIR, "claims.csv"), index=False)
financial_df.to_csv(os.path.join(RAW_DIR, "financial_summary.csv"), index=False)

# -----------------------------
# 14. Quick checks
# -----------------------------
print("Files saved to:", RAW_DIR)
print()
print("Shapes:")
print("customers:", customers_df.shape)
print("vehicles:", vehicles_df.shape)
print("policies:", policies_df.shape)
print("driving_behavior:", driving_df.shape)
print("support_tickets:", tickets_df.shape)
print("claims:", claims_df.shape)
print("financial_summary:", financial_df.shape)

print()
print("Sample counts:")
print("Unique customers in vehicles:", vehicles_df["customer_id"].nunique())
print("Unique customers in policies:", policies_df["customer_id"].nunique())
print("Unique policies in tickets:", tickets_df["policy_id"].nunique())
print("Unique policies in claims:", claims_df["policy_id"].nunique())

Files saved to: C:\Users\thebe\OneDrive\Desktop\IsuranceDA\data_raw

Shapes:
customers: (1100, 18)
vehicles: (1200, 14)
policies: (1260, 21)
driving_behavior: (12896, 17)
support_tickets: (2418, 15)
claims: (260, 13)
financial_summary: (1260, 10)

Sample counts:
Unique customers in vehicles: 1077
Unique customers in policies: 1077
Unique policies in tickets: 1066
Unique policies in claims: 235
